# LLM as a Judge – Automated Evaluation of Responses

**Author:** Ibrahim   
**Environment:** Google Colab / Python 3

## Overview
This notebook turns Groq’s Llama 3.3 70B into an automatic evaluator. Given a question, a reference (ground truth) answer, and a candidate answer, the judge outputs a JSON object with scores for correctness, relevance, clarity, and conciseness (each 1–5) plus a short justification. This technique is used by AlpacaEval, MT‑Bench, and RAGAS.

## What You Will Build
- A function that prompts the judge LLM with a structured scoring rubric.
- Parsing and validation of the JSON output.
- Batch evaluation of multiple (question, ref, candidate) triples.
- Interactive mode – score your own candidate answers.

## Why This Matters
Automated evaluation is critical for A/B testing LLM prompts, fine‑tuned models, and RAG pipelines. It replaces expensive and slow human evaluation.

## Requirements
- **Groq API key** (free from [console.groq.com](https://console.groq.com))

---

**© 2026 Ibrahim – Automated LLM response evaluation.**

### Install & Imports

In [1]:
!pip install -q groq

import json
from getpass import getpass
from groq import Groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 1.2 MB/s eta 0:00:00


### API Key & Groq Client

In [2]:
GROQ_API_KEY = getpass("Enter your Groq API key: ")
client = Groq(api_key=GROQ_API_KEY)
MODEL = "llama-3.3-70b-versatile"
print("Groq client ready.")

Enter your Groq API key: ··········
Groq client ready.


### Judge Function

In [3]:
def judge_response(question, reference_answer, candidate_answer):
    prompt = f"""
You are an impartial expert evaluator. Score the candidate answer on a 1‑5 scale for each criterion.
Return ONLY valid JSON with this structure:
{{
    "correctness": integer (1=poor, 5=excellent),
    "relevance": integer,
    "clarity": integer,
    "conciseness": integer,
    "explanation": "short text"
}}

Question: {question}
Reference (ground truth) answer: {reference_answer}
Candidate answer: {candidate_answer}

JSON output:
"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=512,
        response_format={"type": "json_object"}
    )
    content = response.choices[0].message.content
    try:
        scores = json.loads(content)
        return True, scores
    except json.JSONDecodeError:
        return False, content

### Example 1: Good Candidate

In [4]:
question = "What is the capital of France?"
reference = "Paris"
candidate = "Paris is the capital of France."

success, result = judge_response(question, reference, candidate)
if success:
    print(" Good candidate evaluation:")
    print(json.dumps(result, indent=2))
else:
    print(" Failed:", result)

 Good candidate evaluation:
{
  "correctness": 5,
  "relevance": 5,
  "clarity": 5,
  "conciseness": 4,
  "explanation": "The answer is accurate and directly addresses the question."
}


### Example 2: Poor Candidate

In [5]:
question = "Explain what machine learning is."
reference = "Machine learning is a subset of artificial intelligence that enables systems to learn from data without explicit programming."
candidate = "It's about computers learning stuff."

success, result = judge_response(question, reference, candidate)
if success:
    print(" Poor candidate evaluation:")
    print(json.dumps(result, indent=2))
else:
    print(" Failed:", result)

 Poor candidate evaluation:
{
  "correctness": 2,
  "relevance": 4,
  "clarity": 3,
  "conciseness": 5,
  "explanation": "The answer is brief and related to the topic, but lacks specificity and technical accuracy."
}


### Batch Evaluation

In [6]:
test_cases = [
    ("What is 2 + 2?", "4", "Four", "Good numeric reasoning"),
    ("What is the main ingredient in guacamole?", "Avocado", "Avocado is the main ingredient.", "Perfect answer"),
    ("Who wrote 'Romeo and Juliet'?", "William Shakespeare", "Shakespeare", "Correct but brief")
]

print(" Batch Evaluation Results:\n")
for i, (q, ref, cand, _) in enumerate(test_cases, 1):
    success, scores = judge_response(q, ref, cand)
    if success:
        avg = sum(scores[c] for c in ["correctness","relevance","clarity","conciseness"]) / 4
        print(f"{i}. Q: {q[:50]}... | Avg score: {avg:.1f}")
    else:
        print(f"{i}. Failed to evaluate.")

 Batch Evaluation Results:

1. Q: What is 2 + 2?... | Avg score: 5.0
2. Q: What is the main ingredient in guacamole?... | Avg score: 4.8
3. Q: Who wrote 'Romeo and Juliet'?... | Avg score: 4.8


### Interactive Evaluation

In [7]:
print("LLM‑as‑a‑Judge Interactive")
print("Provide your own question, reference answer, and candidate.")
print("Type 'exit' for question to quit.\n")

while True:
    q = input(" Question (or 'exit'): ").strip()
    if q.lower() == "exit":
        break
    if not q:
        continue
    ref = input(" Reference answer: ").strip()
    cand = input(" Candidate answer: ").strip()
    if not ref or not cand:
        print("Both reference and candidate required.\n")
        continue
    success, result = judge_response(q, ref, cand)
    if success:
        print("\n Evaluation:")
        print(json.dumps(result, indent=2))
    else:
        print(f"\n Error: {result}")

LLM‑as‑a‑Judge Interactive
Provide your own question, reference answer, and candidate.
Type 'exit' for question to quit.

 Question (or 'exit'): What is the capital of France?
 Reference answer: Paris
 Candidate answer: Paris is the capital of France.

 Evaluation:
{
  "correctness": 5,
  "relevance": 5,
  "clarity": 5,
  "conciseness": 4,
  "explanation": "The answer is accurate, directly relevant, and clear, but includes unnecessary words."
}
 Question (or 'exit'): exit


### Final Summary

In [9]:
print("LLM as a Judge - COMPLETED")
print("Author: Ibrahim")
print(" LLM evaluates correctness, relevance, clarity, conciseness.")
print(" Structured JSON output ready for automated pipelines.")
print(" Can be used for RAG, prompt tuning, or model comparison.")
print(" Saves human evaluation effort.")

LLM as a Judge - COMPLETED
Author: Ibrahim
 LLM evaluates correctness, relevance, clarity, conciseness.
 Structured JSON output ready for automated pipelines.
 Can be used for RAG, prompt tuning, or model comparison.
 Saves human evaluation effort.
